<a href="https://colab.research.google.com/github/Lavish05563/california-housing-regression/blob/main/AI_ML_Task3_Model_Validation_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# AI_M nnuL_Task3_Model_Validation_Tuning.ipynb
# Objective: Overfitting control, Cross-Validation, and Hyperparameter Tuning
# ==============================================================================

# Step 1: Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Step 2: Load and Prepare Dataset
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename("HousePrice")], axis=1)
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

# Step 3: Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 4: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# --- TASK COMPLIANCE EVALUATION ---

# 1. Baseline Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

# 2. Baseline Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_r2 = r2_score(y_test, ridge_pred)

# Step 5: Detect Overfitting (Unconstrained Decision Tree)
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)
tree_train_rmse = np.sqrt(mean_squared_error(y_train, tree.predict(X_train)))
tree_test_rmse = np.sqrt(mean_squared_error(y_test, tree.predict(X_test)))

# Step 6: Cross-Validation
cv_scores = cross_val_score(
    tree, X_scaled, y,
    scoring="neg_mean_squared_error",
    cv=5
)
cv_rmse = np.mean(np.sqrt(-cv_scores))

# Step 7: Hyperparameter Tuning Using GridSearchCV
param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10]
}
grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    scoring="neg_mean_squared_error",
    cv=5
)
grid.fit(X_train, y_train)
best_params = grid.best_params_

# Step 8: Evaluate Optimized Model
best_tree = grid.best_estimator_
tuned_pred = best_tree.predict(X_test)
tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_pred))
tuned_r2 = r2_score(y_test, tuned_pred)

# Step 9: Model Comparison Summary Table
results = {
    "Model": ["Linear Regression ki", "Ridge Regression", "Unconstrained Decision Tree", "Tuned Decision Tree"],
    "RMSE": [lr_rmse, ridge_rmse, tree_test_rmse, tuned_rmse],
    "R2 Score": [lr_r2, ridge_r2, r2_score(y_test, tree.predict(X_test)), tuned_r2]
}
summary_df = pd.DataFrame(results)
print("\n=== Model Comparison Table ===")
print(summary_df.to_string(index=False))
print(f"\nBest GridSearch Parameters: {best_params}")


=== Model Comparison Table ===
                      Model     RMSE  R2 Score
       Linear Regression ki 0.745581  0.575788
           Ridge Regression 0.745554  0.575819
Unconstrained Decision Tree 0.703045  0.622811
        Tuned Decision Tree 0.645430  0.682099

Best GridSearch Parameters: {'max_depth': 10, 'min_samples_split': 10}
